# Notebook 2: Huấn luyện và đánh giá RoomNet-Lite

Chạy training pipeline và ghi lại metrics để đưa vào báo cáo.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
from pathlib import Path
import yaml
import torch

from smarthome.config import load_config
from smarthome.preprocessing import RSSIPreprocessor
from smarthome.dataset import RoomWindowDataset
from smarthome.models import RoomNetLite
from smarthome.metrics import room_metrics
from smarthome.utils import set_seed

In [ ]:
ROOT = Path(__file__).resolve().parents[1]
cfg = load_config(ROOT / "config.yaml")
set_seed(int(cfg["seed"]))

In [ ]:
import pandas as pd

train_df = pd.read_csv(ROOT / "data" / "processed" / "room_train.csv")
val_df = pd.read_csv(ROOT / "data" / "processed" / "room_val.csv")
test_df = pd.read_csv(ROOT / "data" / "processed" / "room_test.csv")

preprocessor = RSSIPreprocessor.fit(train_df)
rooms = cfg["rooms"]
window = int(cfg["data"]["window_size"])

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = RoomNetLite(
    in_channels=preprocessor.feature_dim,
    num_rooms=len(rooms),
    hidden_channels=int(cfg["room_model"]["hidden_channels"]),
    embedding_dim=int(cfg["room_model"]["embedding_dim"]),
    dropout=float(cfg["room_model"]["dropout"]),
).to(device)

state = torch.load(ROOT / "models" / "roomnet_lite.pt", map_location=device)
model.load_state_dict(state)
model.eval()

In [ ]:
test_ds = RoomWindowDataset(test_df, preprocessor, rooms, window_size=window)
loader = torch.utils.data.DataLoader(test_ds, batch_size=64)

all_preds, all_true = [], []
with torch.no_grad():
    for x, y, _ in loader:
        logits, _ = model(x.to(device).permute(0, 2, 1))
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y.numpy())

m = room_metrics(all_true, all_preds, rooms)
print("Accuracy:", m["accuracy"])
print("Macro F1:", m["macro_f1"])